# Translating an Endangered Language with LLMs

### *A hands-on tour of the Yaduha framework and the LLM-RBMT paradigm*

This notebook is a guided tour of the research that built an English to **Owens Valley Paiute** translator. It is written for two audiences at once:

- **Linguists, language teachers, and community members**: you do not need to be a programmer. Read the explanations and run the cells with **Shift + Enter**. The code is short and labeled. You will see *why* general-purpose AI translators fail for endangered languages, and *how* the approach in this notebook fixes that.
- **Programmers and ML researchers**: every claim is backed by runnable code that uses the actual `yaduha` and `yaduha-ovp` packages. You can fork any cell and experiment.

By the end you will have:

1. Tried to translate English into Owens Valley Paiute with a state-of-the-art LLM directly and seen it produce nonsense.
2. Understood the grammar of Owens Valley Paiute well enough to recognize a correct sentence.
3. Watched a Pydantic data model **act as a grammar** building grammatically correct sentences by construction.
4. Run the full **LLM-RBMT pipeline**: an LLM that *uses* the grammar instead of inventing one.
5. Run a small experiment comparing the naive approach with the structured approach on the same inputs.

## 0. Setup

This notebook expects two local Python packages to be installed (the framework `yaduha` and the language pack `yaduha-ovp`). If you opened this notebook from inside `kubishi/translation/`, that has already been done for you with [uv](https://docs.astral.sh/uv/).

If you want to set up the environment yourself:

```bash
# from the kubishi/translation/ directory
uv sync
uv run jupyter lab tutorial.ipynb
```

To run the LLM cells you need an **OpenAI API key**. If you do not have one, you can still run the deterministic cells &mdash; we will mark which is which.

Put your key in a `.env` file next to this notebook:

```
OPENAI_API_KEY=sk-...
```

Then run the cell below.

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from this directory

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

# Pre-declare the LLM-backed objects so downstream cells degrade gracefully
# whether or not you have a key, and whether or not you ran the cells in order.
agent = None
pipeline = None
instructions = None

if OPENAI_API_KEY:
    print("OpenAI key loaded — LLM cells will run.")
else:
    print("No OPENAI_API_KEY found — LLM cells will be skipped, but everything else still works.")

if OPENAI_BASE_URL:
    print("OpenAI base URL loaded: " + OPENAI_BASE_URL)
else:
    print("No OPENAI_BASE_URL found — using default OpenAI API base URL.")

print(f"Using model: {OPENAI_MODEL}")


OpenAI key loaded — LLM cells will run.
No OPENAI_BASE_URL found — using default OpenAI API base URL.
Using model: gpt-4o-mini


## 1. Background: Owens Valley Paiute

**Owens Valley Paiute** (OVP, also *Eastern Mono* or *Monache*; ISO code `mnr`) is an Indigenous language of the Numic group of the Uto-Aztecan family, spoken in the Owens Valley region of eastern California.

Some facts that shape the rest of this notebook:

- It is **critically endangered**: 37 to 41 fluent speakers were reported in 1994, and the number is smaller now.
- There is **no publicly available parallel corpus** of OVP and English &mdash; nothing like the millions of aligned sentences that train Google Translate.
- Mistranslations are not just embarrassing &mdash; they can spread through learner materials and *erode* the language being revitalized.

This is what researchers call an **extremely low-resource** (or "no-resource") language. Almost every familiar machine translation technique &mdash; statistical MT, neural MT, fine-tuning a large model &mdash; assumes you have at least a few thousand aligned sentences. We have none.

So the question is: **can we translate into OVP at all, using only what is available &mdash; a dictionary, a grammar description, and a handful of example sentences?**

## 2. The naive baseline: just ask an LLM

Modern LLMs translate dozens of high-resource languages almost perfectly. A reasonable first instinct is: *let me just ask GPT to translate English into Owens Valley Paiute.*

Let's try it. The cell below sends a few English sentences to the model with no special prompting and shows what comes back.

> **If you do not have an API key**, skip this cell &mdash; we have included representative outputs in the markdown right after, so you can still follow along.

In [4]:
from openai import OpenAI

NAIVE_SENTENCES = [
    "I see the dog.",
    "The coyote ran.",
    "You are eating the apple.",
]

if OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)
    for english in NAIVE_SENTENCES:
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            temperature=0.0,
            messages=[
                {"role": "system", "content": "You are a translator from English to Owens Valley Paiute (also called Eastern Mono, ISO code mnr). Respond with ONLY the translation."},
                {"role": "user", "content": english},
            ],
        )
        print(f"EN:  {english}")
        print(f"OVP: {(resp.choices[0].message.content or '').strip()}\n")
else:
    print("(no API key — see the discussion below for representative output)")


EN:  I see the dog.
OVP: Nǫgǫnǫgǫn ǫnǫ.

EN:  The coyote ran.
OVP: Nǫ́ǫtǫ́kǫ́ nǫ́ǫtǫ́.

EN:  You are eating the apple.
OVP: Nǫ́ǫnǫ́ nǫ́ǫnǫ́ kǫ́ǫn.



### What just happened?

If you ran the cell, you saw output that *looks* like a translation. It usually is not.

The model has seen Wikipedia pages and sparse linguistic notes that mention Paiute, but it has never seen aligned English/OVP sentences during training. So it does what LLMs always do under those conditions: it **hallucinates fluently**. The output is in a script that looks vaguely Numic, the words have plausible shapes &mdash; and almost none of it is grammatical.

A native or intermediate speaker reviewing this output would tell you: the verbs are missing tense suffixes, the subject and object suffixes are wrong or swapped, the morphology that distinguishes "this dog" from "that dog" is missing, and several "words" are simply invented.

For a critically endangered language, **fluent-looking nonsense is worse than nothing**. It pollutes learning materials with plausible-sounding errors that are very hard to catch without a fluent speaker, and fluent speakers are exactly what we do not have.

So we need a different approach. Before we can describe it, we need to look at OVP grammar.

## 3. I bet YOU can translate into OVP

The vast majority of you in the audience do not speak any Owens Valley Paiute. Regardless, I am confident that you can translate simple English sentences into OVP with near-perfect accuracy by the end of this section. You just need a little context.

### 3.1 Word Banks

**Word order**: subject, then object, then verb. With a pronoun object the verb moves before the subject.

**Subject suffixes** (attach to a noun used as subject):

| proximity | suffix |
|-----------|--------|
| proximal (this/these) | `-ii` |
| distal (that/those)   | `-uu` |

**Object suffixes** (attach to a noun used as object — depends on whether the OVP word ends in `'`):

| proximity | ends in `'` | does not end in `'` |
|-----------|-------------|---------------------|
| proximal  | `-eika`     | `-neika`            |
| distal    | `-uka`      | `-noka`             |

**Tense / aspect suffixes** (attach to the verb):

| tense | suffix |
|-------|--------|
| past simple | `-ku` |
| present simple | `-dü` |
| present / past continuous | `-ti` |
| present perfect | `-pü` |
| future simple | `-wei` |

**Object-pronoun prefix on the verb** (only on transitive verbs with an object):

| object | prefix |
|--------|--------|
| 3rd person proximal singular (this one) | `a-` |
| 3rd person distal singular (that one)   | `u-` |
| 3rd person proximal plural (these)      | `ai-` |
| 3rd person distal plural (those)        | `ui-` |

**Lenition** (when an object prefix is added, the verb's first consonant softens):

`p → b`, `t → d`, `k → g`, `s → z`, `m → w̃`

**Mini dictionary** (you'll need a subset of these):

| English | OVP | English | OVP |
|---------|-----|---------|-----|
| I (subj) | nüü | sleep | üwi |
| you (subj) | üü | run | poyoha |
| we two (subj) | taa | see | puni |
| coyote | isha' | eat | tüka |
| dog | ishapugu | hit | kwati |
| apple | aaponu' | hear | naka |
| mountain | toyabi | chase | naki |

### 3.2 Worked example

**English:** *I ate that apple.*

1. Subject = "I" → pronoun `nüü`.
2. Object = "that apple" → distal singular. `aaponu'` ends in `'`, so the object suffix is `-uka`. Object becomes `aaponu'-oka`.
3. Verb = "ate" = `tüka`. The object is third-person distal singular, so the object prefix is `u-`, and lenition turns `t` into `d`: stem becomes `u-düka`. Past simple suffix `-ku` gives `u-düka-ku`.
4. Word order is subject–object–verb.

**Result:** `nüü aaponu'-oka u-düka-ku`

### 3.3 Now you try

Work these out on paper (or in your head), then run the cell below to reveal the expected answers.

1. *I am sleeping.*
2. *That coyote ran.*
3. *I will see the dog.*


In [5]:
from tutorial_excercises import EXERCISES

for i, ex in enumerate(EXERCISES, start=1):
    print(f"{i}. {ex['english']}")
    print(f"   → {ex['expected']}")
    print()


1. I am sleeping.
   → nüü üwi-ti

2. That coyote ran.
   → isha'-uu poyoha-ku

3. I will see the dog.
   → nüü ishapugu-noka u-buni-wei



### 3.4 I just prompt-engineered you

If you got through even one of those, congratulations, you have just been prompt-engineered! Look back at what you were given:

- A **system prompt**: the reference card in §3.1 with the rules and a small word bank.
- A **one-shot example**: the worked translation of *I ate that apple* in §3.2, broken down step by step.
- A **user query**: each English sentence in §3.3.

That is exactly the structure of a prompt to an LLM. Rules and examples were presented to you and you applied them in context.

## 4. Now let me prompt-engineer the LLM

Recall what I gave you in §3:

- a system prompt (the reference card),
- a few-shot example (the worked translation of *I ate that apple*),
- a sequence of user queries (the three English sentences).

This time I am going to give the *exact same materials* to GPT, with the same English sentences you just translated, and we will see what comes out. This is what `yaduha` calls the **Instructions translator** &mdash; the entire grammar lives in the system prompt, and the LLM is asked to apply it directly.


In [6]:
INSTRUCTIONS = """You are a translator from English to Owens Valley Paiute (OVP).

Word order: subject, then object, then verb. With a pronoun object, the verb moves before the subject.

Subject suffixes (attach to a noun used as subject):
  proximal (this/these): -ii
  distal (that/those):   -uu

Object suffixes (depend on whether the OVP word ends in '):
  proximal + ends in ': -eika      proximal + does not: -neika
  distal   + ends in ': -uka       distal   + does not: -noka

Tense / aspect suffixes (attach to the verb):
  past simple = -ku, present simple = -dü, present/past continuous = -ti,
  present perfect = -pü, future simple = -wei

Object-pronoun prefix on a transitive verb that has an object:
  3rd-person proximal singular = a-, distal singular = u-,
  3rd-person proximal plural   = ai-, distal plural   = ui-

Lenition (when an object prefix is added, the verb's first consonant softens):
  p -> b, t -> d, k -> g, s -> z, m -> w̃

Mini dictionary:
  I = nüü, you = üü, we two = taa
  coyote = isha', dog = ishapugu, apple = aaponu', mountain = toyabi
  sleep = üwi, run = poyoha, see = puni, eat = tüka, hit = kwati,
  hear = naka, chase = naki

Worked example:
  English: I ate that apple.
  OVP:     nüü aaponu'-uka u-düka-ku

Respond with ONLY the OVP translation. No commentary, no English glosses."""


def translate_with_instructions(english: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user", "content": english},
        ],
    )
    return (resp.choices[0].message.content or '').strip()


if OPENAI_API_KEY:
    for ex in EXERCISES:
        out = translate_with_instructions(ex["english"])
        match = "MATCH" if out == ex["expected"] else "DIFF "
        print(f"EN:        {ex['english']}")
        print(f"LLM:       {out}")
        print(f"Expected:  {ex['expected']}  [{match}]")
        print()
else:
    print("(skipped — needs API key)")


EN:        I am sleeping.
LLM:       nüü üwi-dü
Expected:  nüü üwi-ti  [DIFF ]

EN:        That coyote ran.
LLM:       isha'-uu poyoha-ku
Expected:  isha'-uu poyoha-ku  [MATCH]

EN:        I will see the dog.
LLM:       nüü ishapugu-neika a-puni-wei
Expected:  nüü ishapugu-noka u-buni-wei  [DIFF ]



### 4.1 What just happened

If everything in the sentence is covered by the reference card, the LLM does pretty well. Sometimes it picks a different but still defensible tense. That makes sense: every word, every rule, every example is right there in the context window. The model is doing the same kind of pattern-matching from context that you did.


## 5. Words the prompt doesn't have

Try a sentence whose nouns or verbs are not in the mini dictionary above. The LLM has nothing to fall back on — it has not seen meaningful amounts of OVP during training, and the system prompt does not cover the word.


In [7]:
OUT_OF_VOCAB_SENTENCES = [
    "The cat ran.",
    "I see the bear.",
    "The fish is sleeping.",
]

if OPENAI_API_KEY:
    for english in OUT_OF_VOCAB_SENTENCES:
        print(f"EN:  {english}")
        print(f"OVP: {translate_with_instructions(english)}")
        print()
else:
    print("(skipped — needs API key)")


EN:  The cat ran.
OVP: toyabi-ii isha' poyoha-dü

EN:  I see the bear.
OVP: nüü a'kwi'-neika a-puni-dü

EN:  The fish is sleeping.
OVP: toyabi-ii üwi-dü



### 5.1 Hallucinated vocabulary

Look closely at what came back. The LLM either:

- **kept the English word as-is** (`kidi`, `bear`, `fish`) and hoped nobody would notice,
- **invented a plausible-sounding stem** (something Numic-looking that does not actually exist), or, more rarely,
- **used another word** from the provided workd list.

We cannot tell from the output which of these is happening. To a non-speaker (and to most learners), all three look equally plausible. **Fluent-looking, undetectable hallucination is exactly the failure mode that makes LLMs dangerous for endangered-language work.**

Including the entire vocabulary in the system prompt is not realistic. The dictionary has thousands of entries and grows over time. We need a different strategy.


## 6. How would *you* solve that?

If a word was missing from the reference card I handed you, what would you actually do? You would *go look it up*. Open a tab. Type the English word. Read the entry.

There is a public Owens Valley Paiute dictionary online: **[dictionary.kubishi.com](https://dictionary.kubishi.com)**. It searches by English or by Paiute, and every entry has a definition and (often) example sentences.

Try it: open the site, search for one of the words the LLM choked on above (*cat*, *bear*, *fish*), and confirm that the dictionary has it. You just performed, by hand, the kind of lookup we are about to give the LLM.

This is the intuition behind **retrieval-augmented generation** (RAG): instead of trying to stuff every fact into the prompt, give the model a *tool* and let it look things up on demand.


## 7. Give the LLM a dictionary tool

The Kubishi site is backed by a public REST API documented at **[dictionary.kubishi.com/api/docs](https://dictionary.kubishi.com/api/docs)**. The endpoint we want is `/api/search?q=<english>` &mdash; it returns Paiute words whose English glosses match the query.

We will:

1. Write a Python function that hits that endpoint and returns a small, clean result.
2. Register it as a **tool** the LLM can call (OpenAI's function-calling / tools API).
3. Run translations again and watch the model decide *on its own* when to look a word up.


In [8]:
import httpx

DICT_API = "https://dictionary.kubishi.com/api"


def lookup_word(english_query: str, limit: int = 5) -> list[dict]:
    """Search the Kubishi OVP dictionary for an English term.

    Returns up to `limit` Paiute matches, each with paiute form, gloss, and definition.
    """
    r = httpx.get(f"{DICT_API}/search", params={"q": english_query, "limit": limit}, timeout=15.0)
    r.raise_for_status()
    results = []
    for entry in r.json().get("results", []):
        senses = entry.get("senses") or []
        first = senses[0] if senses else {}
        results.append({
            "paiute": entry.get("lexical_unit"),
            "gloss": first.get("gloss", ""),
            "definition": first.get("definition", ""),
        })
    return results


# Quick smoke test
print("cat   ->", lookup_word("cat",  limit=2))
print("bear  ->", lookup_word("bear", limit=2))
print("fish  ->", lookup_word("fish", limit=2))


cat   -> [{'paiute': "kiidi'", 'gloss': 'cat', 'definition': 'Domestic cat.'}, {'paiute': 'wiheetsiti', 'gloss': 'mountain lion', 'definition': 'Mountain lion, wild cat.'}]
bear  -> [{'paiute': 'pahabichi', 'gloss': 'bear', 'definition': 'Bear, black bear, scientific name: Ursus americanus'}, {'paiute': "ünü'", 'gloss': 'Euro-American', 'definition': 'Euro-American person, white person. Something frightening. This word also has been used to refer to the bear or California Grizzly possibly in place of a more traditional name that is no longer in use or lost.'}]
fish  -> [{'paiute': 'pagwiga', 'gloss': 'fish', 'definition': 'To fish, to collect fish.'}, {'paiute': 'pagwi', 'gloss': 'fish', 'definition': 'Fish, general name for fish.'}]


In [9]:
import json as _json

def translate_with_tools(english: str, max_iters: int = 6) -> str:
    messages = [
        {"role": "system", "content": INSTRUCTIONS + "\n\nWhenever a noun or verb in the input is not in the mini dictionary above, call the lookup_word tool to find its Paiute form. Do not guess."},
        {"role": "user", "content": english},
    ]
    for _ in range(max_iters):
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            temperature=0.0,
            messages=messages,
            tools=[{
                "type": "function",
                "function": {
                    "name": "lookup_word",
                    "description": (
                        "Search the Kubishi Owens Valley Paiute dictionary for an English term. "
                        "Use this whenever a noun or verb in the input is NOT in the mini dictionary "
                        "in your system prompt. Returns Paiute words with their English glosses."
                    ),
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "english_query": {"type": "string", "description": "An English word to look up."},
                            "limit": {"type": "integer", "default": 3, "description": "Max results."},
                        },
                        "required": ["english_query"],
                    },
                },
            }]
        )
        msg = resp.choices[0].message
        # Echo the tool calls so the reader can see what the LLM is doing
        if msg.tool_calls:
            for call in msg.tool_calls:
                args = _json.loads(call.function.arguments)
                print(f"  [tool call] lookup_word({args})")
            messages.append(msg)
            for call in msg.tool_calls:
                args = _json.loads(call.function.arguments)
                try:
                    result = lookup_word(**args)
                except Exception as exc:
                    result = {"error": str(exc)}
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": _json.dumps(result),
                })
            continue
        return (msg.content or "").strip()
    return "(no answer within max_iters)"


if OPENAI_API_KEY:
    for english in OUT_OF_VOCAB_SENTENCES:
        print(f"EN:  {english}")
        result = translate_with_tools(english)
        print(f"OVP: {result}")
        print()
else:
    print("(skipped — needs API key)")


EN:  The cat ran.
  [tool call] lookup_word({'english_query': 'cat'})
OVP: kii'di' -neika poyoha -dü

EN:  I see the bear.
  [tool call] lookup_word({'english_query': 'bear'})
  [tool call] lookup_word({'english_query': 'see'})
OVP: nüü pahabichi-ii a-puni-dü

EN:  The fish is sleeping.
  [tool call] lookup_word({'english_query': 'fish'})
OVP: pagwi-ii üwi-düka



### 7.1 Vocabulary, fixed. Grammar, not so much.

Notice the `[tool call]` lines: the model decided *on its own* that *cat*, *bear*, and *fish* were not in its mini-dictionary, so it queried the live API for them and used the result. This helps address the vocabulary problem. We could let the LLM grow as much vocabulary as it likes by hitting an API on demand.

But push it on harder sentences and the cracks reappear.


In [10]:
COMPLEX_SENTENCES = [
    "The bear is chasing the rabbit through the forest.",
    "We two will see those mountains tomorrow morning.",
    "Yesterday the coyote ate the apple that I had picked.",
    "If the dog runs, the cat will hide.",
]

if OPENAI_API_KEY:
    for english in COMPLEX_SENTENCES:
        print(f"EN:  {english}")
        print(f"OVP: {translate_with_tools(english)}")
        print()
else:
    print("(skipped — needs API key)")


EN:  The bear is chasing the rabbit through the forest.
  [tool call] lookup_word({'english_query': 'bear'})
  [tool call] lookup_word({'english_query': 'rabbit'})
  [tool call] lookup_word({'english_query': 'forest'})
OVP: ii pahabichi kammüna uboggina naki-dü

EN:  We two will see those mountains tomorrow morning.
  [tool call] lookup_word({'english_query': 'mountain'})
  [tool call] lookup_word({'english_query': 'morning'})
OVP: taa toyabi-uu aaŵahusu u-puni-wei

EN:  Yesterday the coyote ate the apple that I had picked.
  [tool call] lookup_word({'english_query': 'yesterday'})
  [tool call] lookup_word({'english_query': 'picked'})
OVP: yongo isha' aaponu'-uka ai-tsoba-neika u-düka-ku

EN:  If the dog runs, the cat will hide.
  [tool call] lookup_word({'english_query': 'cat'})
  [tool call] lookup_word({'english_query': 'hide'})
OVP: ishapugu toyabi poyoha, kiidi' waadzi-wei.



### 7.2 Why this is still not enough

Even with a working dictionary tool, the LLM still:

- forgets the object-suffix on the second of two object nouns,
- skips lenition when there *is* an object prefix,
- invents tense suffixes that do not exist (`-tonu`, `-kawei`),
- silently flattens dependent clauses (*"that I had picked"*, *"if the dog runs"*) instead of splitting the input into simpler sentences the grammar can express.

These are not vocabulary failures; they are **grammar failures**. RAG does not fix them, because RAG is about retrieving facts, not following rules. Adding more tools, more examples, and more clarification to the prompt helps a little &mdash; but the fundamental shape of the failure is the same: the LLM can read rules and *usually* applies them, but "usually" is not good enough for a critically endangered language.

So we change the question. Instead of asking the LLM to *follow* the grammar, what if the LLM could only *speak through* the grammar &mdash; what if the only sentences it could utter were grammatical ones, by construction? That is what the next section is about.


## 8. The big idea: a Pydantic model *is* the grammar

The package `yaduha-ovp` defines OVP grammar as a set of Python classes. The type system encodes (part of) the grammar of the language. The subset is small on purpose: we trade coverage for a guarantee that nothing ungrammatical can ever come out. Below we import the building blocks one at a time.

In [11]:
from yaduha_ovp import (
    Pronoun,
    Proximity,
    Plurality,
    TenseAspect,
    SubjectNoun,
    ObjectNoun,
    TransitiveVerb,
    IntransitiveVerb,
    SubjectVerbSentence,
    SubjectVerbObjectSentence,
)

# Each grammatical category is an enum — the only allowed values are the linguistically-real ones.
print("Proximity values:", [p.value for p in Proximity])
print("Plurality values:", [p.value for p in Plurality])
print("Tense/aspect:    ", [t.value for t in TenseAspect])
print()
print("OVP pronouns:")
for p in Pronoun:
    print(f"  {p.name:25s}  ({p.value})")


Proximity values: ['proximal', 'distal']
Plurality values: ['singular', 'dual', 'plural']
Tense/aspect:     ['past_simple', 'past_continuous', 'present_perfect', 'present_simple', 'present_continuous', 'future_simple']

OVP pronouns:
  I                          (I)
  we_two                     (we (two))
  we_inclusive               (we (inclusive))
  we_exclusive               (we (exclusive))
  you                        (you)
  you_all                    (you (plural))
  he_she_it_proximal         (he/she/it (proximal))
  he_she_it_distal           (he/she/it (distal))
  they_proximal              (they (proximal))
  they_distal                (they (distal))
  reflexive                  (self (reflexive))


### 8.1 A noun is a structured object

A `SubjectNoun` is not a string. It is an object with three required pieces of information that the grammar demands:

- a **head** (the lemma, like `"apple"`),
- a **proximity** (proximal or distal),
- a **plurality** (singular, dual, or plural).

You cannot construct one without choosing all three. That is the grammar enforcing itself.

In [12]:
coyote_distal = SubjectNoun(head="coyote", proximity=Proximity.distal, plurality=Plurality.singular)
print(coyote_distal)            # rendered with the right subject suffix
print(repr(coyote_distal))      # the underlying structured form


head='coyote' possessive_determiner=None proximity=<Proximity.distal: 'distal'> plurality=<Plurality.singular: 'singular'>
SubjectNoun(head='coyote', possessive_determiner=None, proximity=<Proximity.distal: 'distal'>, plurality=<Plurality.singular: 'singular'>)


### 8.2 A verb chooses tense by construction

A `TransitiveVerb` carries a lemma *and* a tense/aspect. There is no way to forget the tense suffix &mdash; the type makes you pick one.

In [13]:
eat_past = TransitiveVerb(lemma="eat", tense_aspect=TenseAspect.past_simple)
see_future = TransitiveVerb(lemma="see", tense_aspect=TenseAspect.future_simple)
print(eat_past)
print(see_future)


lemma='eat' tense_aspect=<TenseAspect.past_simple: 'past_simple'>
lemma='see' tense_aspect=<TenseAspect.future_simple: 'future_simple'>


### 8.3 A sentence is a structured object too

`SubjectVerbObjectSentence` requires a subject, a transitive verb, and an object. Putting them together produces a fully-formed OVP sentence &mdash; with the right suffixes, the right object prefix on the verb, and lenition applied automatically &mdash; just by calling `str(...)`.

In [14]:
sentence = SubjectVerbObjectSentence(
    subject=Pronoun.I,
    verb=TransitiveVerb(lemma="eat", tense_aspect=TenseAspect.past_simple),
    object=ObjectNoun(head="apple", proximity=Proximity.distal, plurality=Plurality.singular),
)

print("Built from structured pieces:")
print(sentence.model_dump_json(indent=2))
print()
print("Rendered to OVP:")
print(" ", sentence)
print()
print("English meaning: 'I ate that apple.'")


Built from structured pieces:
{
  "subject": "I",
  "verb": {
    "lemma": "eat",
    "tense_aspect": "past_simple"
  },
  "object": {
    "head": "apple",
    "possessive_determiner": null,
    "proximity": "distal",
    "plurality": "singular"
  }
}

Rendered to OVP:
  nüü aaponu'-uka u-düka-ku

English meaning: 'I ate that apple.'


Notice what happened in the rendering:

- `nüü` came from the subject pronoun `I`.
- `aaponu'-uka` came from the object noun: lemma `apple` &rarr; OVP stem `aaponu'`, plus the distal-glottal object suffix `-uka`.
- `u-düka-ku` came from the verb: object-prefix `u-` (distal singular) + leniated stem `düka` (from `tüka`) + past-simple suffix `-ku`.

We did not write any of those rules in this cell. They live inside the Pydantic models, encoded once, and applied every time a sentence is rendered.

### 8.4 Try it yourself

Below are a handful of sentences built from different combinations of subject, verb, tense, and object. Each one is rendered through the same grammar machinery you saw above. Edit the list (or add your own rows) and re-run the cell &mdash; whatever you construct is guaranteed to come out grammatical.

In [15]:
EXAMPLES = [
    SubjectVerbObjectSentence(
        subject=Pronoun.I,
        verb=TransitiveVerb(lemma="eat", tense_aspect=TenseAspect.past_simple),
        object=ObjectNoun(head="apple", proximity=Proximity.distal, plurality=Plurality.singular),
    ),
    SubjectVerbObjectSentence(
        subject=Pronoun.you,
        verb=TransitiveVerb(lemma="see", tense_aspect=TenseAspect.future_simple),
        object=ObjectNoun(head="dog", proximity=Proximity.proximal, plurality=Plurality.singular),
    ),
    SubjectVerbObjectSentence(
        subject=SubjectNoun(head="coyote", proximity=Proximity.distal, plurality=Plurality.singular),
        verb=TransitiveVerb(lemma="chase", tense_aspect=TenseAspect.present_continuous),
        object=ObjectNoun(head="mountain", proximity=Proximity.distal, plurality=Plurality.plural),
    ),
    SubjectVerbObjectSentence(
        subject=Pronoun.we_two,
        verb=TransitiveVerb(lemma="hear", tense_aspect=TenseAspect.present_simple),
        object=Pronoun.he_she_it_distal,
    ),
]

for sentence in EXAMPLES:
    print(sentence)


nüü aaponu'-uka u-düka-ku
üü ishapugu-neika a-buni-wei
isha'-uu toyabi-noka ui-naki-ti
u-naka-dü taa


### 8.5 Random grammatically-valid sentences

Because the grammar is encoded in code, we can ask Python to generate as many valid sentences as we like. This can be useful for testing, for building example sets, or just for getting a feel for the language.

In [16]:
for s in SubjectVerbObjectSentence.sample_iter(8):
    print(s)


pagwi-uu wo'abi-neika a-dsibui-wei
küna-ii wo'abi-neika ai-gwati-ti
i-gwana-wei uhuw̃a
na-naka-ti pasohobü-uu
nüü tüba-neika ai-naki-ti
wihi-uu isha'-eika ai-w̃ui-dü
ui-hibi-pü wihi-ii
ta-dama'i-wei taagwa


## 9. The LLM-RBMT pipeline: an LLM that *uses* the grammar

We now have two things:

1. A grammar that **always produces a valid OVP sentence**.
2. An LLM that is great at understanding English but **invents nonsense** when asked to produce OVP directly.

The insight of **LLM-Assisted Rule-Based Machine Translation** (LLM-RBMT) is to combine them:

> Let the LLM make the *choices* (which lemma, which tense, which proximity), and let the rules render the result.

The LLM never writes or interacts with OVP directly. It only writes Pydantic objects. The structured-output feature of modern LLMs guarantees that the object is well-formed.

The full pipeline looks like this:

```
English input
    │
    ▼
[ Sentence simplifier ]   ← LLM call: "I saw him eat the apple."  →  ["He ate the apple.", "I saw him."]
    │
    ▼
[ Structured translator ]  ← LLM call with Pydantic schema → SubjectVerbObjectSentence(...)
    │
    ▼
[ Renderer ]  ← deterministic Python: turn the object into 'isha'-uu a-buni-ku' style strings
    │
    ▼
[ Back-translator ]  ← LLM call: render the structured form back into English to verify meaning
```

Steps 1 and 2 use an LLM but constrain its output. Step 3 is simple python code. Step 4 lets a non-speaker user check whether their input was understood correctly. This is a critical feature when there are no fluent speakers around to verify.

Below we instantiate the actual `PipelineTranslator` from `yaduha`.

In [17]:
from yaduha.agent.openai import OpenAIAgent
from yaduha.translator.pipeline import PipelineTranslator

if OPENAI_API_KEY:
    agent = OpenAIAgent(model=OPENAI_MODEL, api_key=OPENAI_API_KEY)
    pipeline = PipelineTranslator.from_language("ovp", agent=agent)
    print("Pipeline translator ready.")
else:
    pipeline = None
    print("No API key — skipping; results from a previous run shown below.")


Pipeline translator ready.


### 9.1 Run the pipeline on the same sentences that broke the naive LLM

In [18]:
def show_translation(t):
    print(f"EN  : {t.source}")
    print(f"OVP : {t.target}")
    if t.back_translation:
        print(f"BACK: {t.back_translation.source}")
    print(f"      ({t.translation_time:.2f}s, {t.prompt_tokens} prompt + {t.completion_tokens} completion tokens)")
    print()


if pipeline is not None:
    for english in NAIVE_SENTENCES:
        show_translation(pipeline.translate(english))
else:
    print("(skipped — needs API key)")


/workspaces/translation_tutorial/.venv/lib/python3.12/site-packages/pydantic/main.py:542: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=TargetSentenceList(senten...ingular: 'singular'>))]), input_type=TargetSentenceList])
  return self.__pydantic_serializer__.to_json(


EN  : I see the dog.
OVP : Nüü ishapugu-noka u-buni-dü.
BACK: I see the dog.
      (1.59s, 1553 prompt + 58 completion tokens)



/workspaces/translation_tutorial/.venv/lib/python3.12/site-packages/pydantic/main.py:542: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=TargetSentenceList(senten...mple: 'past_simple'>))]), input_type=TargetSentenceList])
  return self.__pydantic_serializer__.to_json(


EN  : The coyote ran.
OVP : Isha'-uu poyoha-ku.
BACK: The coyote ran.
      (1.52s, 1553 prompt + 55 completion tokens)

EN  : You are eating the apple.
OVP : Üü aaponu'-uka u-düka-ti.
BACK: You are eating the apple.
      (1.65s, 1554 prompt + 60 completion tokens)



### 9.2 Look inside the pipeline

The translator did three things internally for each input sentence:

1. Turned the English into a list of `SubjectVerbSentence` / `SubjectVerbObjectSentence` objects (the LLM, constrained by the Pydantic schema).
2. Called `str(...)` on each (pure Python, deterministic).
3. Asked an LLM to read the structured form back into English, so we have something a non-speaker can verify.

We can call the inner step directly to see the structured intermediate.

In [19]:
from yaduha.tool.english_to_sentences import EnglishToSentencesTool

if pipeline is not None:
    tool = EnglishToSentencesTool(agent=agent, SentenceType=(SubjectVerbSentence, SubjectVerbObjectSentence))
    response = tool("I will see the dog.")
    for sent in response.content.sentences:
        print("Structured:", sent.model_dump_json(indent=2))
        print("Rendered  :", sent)
        print()
else:
    print("(skipped — needs API key)")


Structured: {
  "subject": "I",
  "verb": {
    "lemma": "see",
    "tense_aspect": "future_simple"
  },
  "object": {
    "head": "dog",
    "possessive_determiner": null,
    "proximity": "distal",
    "plurality": "singular"
  }
}
Rendered  : nüü ishapugu-noka u-buni-wei



### 9.3 Sentence simplification

What about input the grammar *cannot* directly express, like *"I saw him eat the apple"*? OVP does not have an equivalent of English's accusative-with-infinitive construction.

The pipeline handles this by asking the LLM to **simplify**: split the input into two simpler sentences that the grammar *can* express.

> *"I saw him eat the apple"* &rarr; *"He ate the apple."* + *"I saw him."*

The information loss is real but explicit, and the user can see it via the back-translation.

In [20]:
if pipeline is not None:
    show_translation(pipeline.translate("I saw him eat the apple."))
    show_translation(pipeline.translate("She laughed quietly at his silly joke."))
else:
    print("(skipped — needs API key)")


EN  : I saw him eat the apple.
OVP : A-buni-ku nüü. Mahu aaponu'-uka u-düka-ku.
BACK: I saw him/her/it. He/She/It ate the apple.
      (2.63s, 1555 prompt + 95 completion tokens)

EN  : She laughed quietly at his silly joke.
OVP : Mahu nishua'i-ku.
BACK: He/She/It laughed.
      (1.28s, 1556 prompt + 37 completion tokens)



## 10. Mini experiment: Pipeline vs. Instructions

In [21]:
from yaduha.translator.instructions import InstructionsTranslator

if OPENAI_API_KEY:
    instructions = InstructionsTranslator.from_language("ovp", agent=agent)
    print("Instructions translator ready.")
else:
    instructions = None


Instructions translator ready.


In [22]:
EXPERIMENT_SENTENCES = [
    "The dog sees the cat.",
    "I will eat the apple.",
    "We are running.",
    "The coyote chased that rabbit.",
    "You read the book.",
]

if pipeline is not None and instructions is not None:
    for english in EXPERIMENT_SENTENCES:
        print(f"=== {english} ===")
        p = pipeline.translate(english)
        i = instructions.translate(english)
        print(f"  Pipeline    : {p.target}")
        print(f"    back→EN   : {p.back_translation.source if p.back_translation else '-'}")
        print(f"  Instructions: {i.target}")
        print()
else:
    print("(skipped — needs API key)")


=== The dog sees the cat. ===
  Pipeline    : Ishapugu-uu kidi'-uka u-buni-dü.
    back→EN   : That dog sees that cat.
  Instructions: Owens Valley Paiute: ishapugu-uu kidi'-neika a-puni-dü

=== I will eat the apple. ===
  Pipeline    : Nüü aaponu'-uka u-düka-wei.
    back→EN   : I will eat that apple.
  Instructions: nüü aaponu-neika i-tüka-wei

=== We are running. ===


/workspaces/translation_tutorial/.venv/lib/python3.12/site-packages/pydantic/main.py:542: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=TargetSentenceList(senten...present_continuous'>))]), input_type=TargetSentenceList])
  return self.__pydantic_serializer__.to_json(


  Pipeline    : Nüügwa poyoha-ti.
    back→EN   : We are running.
  Instructions: taa poyoha-dü

=== The coyote chased that rabbit. ===
  Pipeline    : Isha'-uu [rabbit]-noka u-naki-ku.
    back→EN   : That coyote chased the rabbit.
  Instructions: isha'-uu naki-wei tabuutsi'-uka

=== You read the book. ===
  Pipeline    : Üü [book]-noka u-nia-dü.
    back→EN   : You read the book.
  Instructions: üü nobi-noka ü-nia-dü



### What to look for

- **Pipeline** outputs are guaranteed to follow the schema in section 3 (subject suffix, object suffix, object-pronoun prefix, lenition, tense suffix). If you read the structured form, you can verify the grammar mechanically.
- **Instructions** outputs are often close to right but quietly drop a suffix, fail to leniate, or use a wrong object-pronoun prefix. To a non-speaker the difference is invisible.

In our published evaluation on 150 sentences across six grammatical categories, the Pipeline translator achieved the highest overall translation quality, while the Instructions translator was the easiest to implement but least reliable for production use.

## 11. Recap and where to go next

What we did:

1. Confirmed that asking a state-of-the-art LLM to translate into OVP directly produces fluent-sounding nonsense.
2. Translated a few sentences by hand using a reference card and noticed that the exercise was structured exactly like a prompt: system prompt, few-shot example, user query.
3. Handed the same reference card to GPT (the **Instructions translator**) and watched it succeed on in-context vocabulary and fail on out-of-context vocabulary.
4. Gave the LLM a **dictionary tool** (RAG) and saw that retrieval helps with the vocabulary problem but grammar errors persist on complex sentences. RAG retreives useful information but it cannot enforce rules.
5. Saw that the **Pydantic models in `yaduha-ovp` *are* the grammar**. Constructing a valid object is exactly constructing a valid sentence.
6. Ran the **LLM-RBMT pipeline**, which lets an LLM make grammatical *choices* while the grammar enforces correctness deterministically.
7. Compared the structured pipeline against a prompt-only baseline.

### For the linguist or community member

The Yaduha framework is designed so that adding a new language means writing a new language pack, a Pydantic schema describing your grammar, a vocabulary list, and example sentences. No machine learning training required and no parallel corpus required. The grammar lives entirely in code that a linguist and a programmer can read together.

### For the programmer or researcher

- The framework: [`yaduha`](./yaduha/): agents, translators, evaluators, language loader.
- The OVP language pack: [`yaduha-ovp`](./yaduha-ovp/): a worked example to copy.
- The papers: [LLM-Assisted Rule Based Machine Translation for Low/No-Resource Languages](https://aclanthology.org/2024.americasnlp-1.9.pdf) (the original LLM-RBMT paper) and [Comparing LLM-Based Translation Approaches for Extremely Low-Resource Languages](https://aclanthology.org/2026.loresmt-1.4.pdf) (the systematic evaluation across five translator strategies).

To build your own language pack, start by copying `yaduha-ovp/yaduha_ovp/` and editing `vocab.py` and the sentence classes in `__init__.py`. The `LanguageLoader.validate_language(...)` helper will check that you have all the required pieces.

> *Maanohoobüü! Thank you!*
